In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "command-r7b:7b-12-2024-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos comunes a todos los modelos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"

# Conectar al servidor local de Ollama
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# Consultar características publicadas por Ollama
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: command-r7b:7b-12-2024-q4_K_M
Identificador: ff4e9696ef9f
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'tools']
Thinking: None
Parámetros: stop                           "<|START_OF_TURN_TOKEN|>"
stop                           "<|END_OF_TURN_TOKEN|>"
stop                           "<|END_RESPONSE|>"

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-COMMANDR7B-01"

# 1. Caso diagnóstico común
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# Este modelo no dispone de modo thinking
solicitud.pop("think", None)

# Parámetros de evaluación
solicitud["options"] = {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0
}

# Mantener los prompts V3 originales
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# 3. Verificaciones
assert configuracion["prompt_version"] == "V3"
assert MODELO == "command-r7b:7b-12-2024-q4_K_M"
assert "completion" in detalles.get("capabilities", [])
assert "thinking" not in detalles.get("capabilities", [])
assert sha_casos == configuracion["cases_sha256"]

# Evitar duplicar el diagnóstico
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "command_r7b_directa_v1",
    "prompt_version": "V3",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar inferencia local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar resultado sin sobrescribir
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar resultados
print("Caso:", CASO_ID)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)

        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Caso: DIAG-COMMANDR7B-01
Estado: ok
Tiempo cliente: 7.48 s
Tokens entrada: 723
Tokens generados: 229
Terminación: stop
Razonamiento separado: False
total_duration 7.479 s
load_duration 3.345 s
prompt_eval_duration 0.257 s
eval_duration 3.866 s

RESPUESTA FINAL:

**Evidencia:**

* [E1] Autor X tiene un artículo sobre bases de datos académicas.
* [E2] Autor Y tiene un artículo sobre recuperación de información.

**Interpretación:**

La similitud temática entre los artículos de Autor X y Autor Y (bases de datos académicas y recuperación de información) sugiere que ambos autores podrían tener intereses y conocimientos relacionados. Sin embargo, esto no demuestra que hayan trabajado juntos, compartido un proyecto o colaboren en el futuro.

**Limitaciones:**

* La similitud temática es un factor importante, pero no es suficiente para demostrar una colaboración previa.
* La falta de datos de colaboración previa limita la capacidad de inferir una relación de trabajo entre los autores.

**Concl

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "command-r7b:7b-12-2024-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-COMMANDR7B-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert "think" not in diagnostico["solicitud"]

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0
}

assert diagnostico["sha256_casos"] == sha_casos
assert diagnostico["sha256_configuracion"] == sha_config

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Comprobar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar casos
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "command_r7b_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Reutilizar la solicitud del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Cambiar únicamente el caso sintético
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Preparar registro
        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "command_r7b_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar inmediatamente cada ejecución
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 355 tokens | 6.682 s
S-006 | ok | 226 tokens | 4.179 s
S-007 | ok | 281 tokens | 5.17 s
S-008 | ok | 279 tokens | 4.864 s
S-009 | ok | 300 tokens | 5.233 s
S-010 | ok | 304 tokens | 5.322 s

Archivo actualizado: resultados.jsonl
